# All the imports

In [347]:
import numpy as np
import os
import docx
import re
import spacy
import pdfplumber
from datetime import datetime


In [348]:
import os
print(os.listdir())

['resume_3.pdf', 'renamed_files', 'resume_7.pdf', '.virtual_documents', 'resume_4.pdf', 'dataset', 'resume_2.pdf', 'resume_6.pdf', 'jd.pdf', 'resume_5.pdf', 'resume_1.pdf']


# Import Dataset from Kaggle

In [349]:
dataset_path = "/kaggle/input/datasets/adityadaveddd/nlp-dataset"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025065 - Keyur Padiya.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2024065_SDE_RESUME.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025047_GauravRajpurohit.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025059 - Kartavyakumar Patel.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025013 - Aditya Dave.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025069 (2).pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/MT2025085.pdf
/kaggle/input/datasets/adityadaveddd/nlp-dataset/JD.pdf


# Change resume file names into meaningful names

In [350]:
import os
import shutil

input_path = "/kaggle/input/datasets/adityadaveddd/nlp-dataset"
output_path = "/kaggle/working/dataset"

os.makedirs(output_path, exist_ok=True)

resume_count = 1

for root, dirs, files in os.walk(input_path):
    for file in sorted(files):
        old_path = os.path.join(root, file)

        if "jd" in file.lower():
            new_name = "JD.pdf"
        else:
            new_name = f"resume_{resume_count}.pdf"
            resume_count += 1

        new_path = os.path.join(output_path, new_name)
        shutil.copy(old_path, new_path)
print("Dataset prepared successfully!")

Dataset prepared successfully!


# Convert uploaded files into raw text

In [351]:
# ----------- PDF Extraction -----------
def extract_text_from_pdf(file_path):
    text = ""
    
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    
    return text


# ----------- DOCX Extraction -----------
def extract_text_from_docx(file_path):
    doc = docx.Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs])
    return text


# ----------- Main Extractor -----------
def extract_text(file_path):
    if file_path.lower().endswith(".pdf"):
        return extract_text_from_pdf(file_path)
    elif file_path.lower().endswith(".docx"):
        return extract_text_from_docx(file_path)
    else:
        return ""

## Basic Cleaning

In [352]:
def clean_text(text):
    text = text.lower()
    
    # 🔥 Step 1: Protect decimal numbers
    text = re.sub(r'(\d)\.(\d)', r'\1DOT\2', text)

    # Step 2: Remove unwanted characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # Step 3: Restore decimal points
    text = text.replace("DOT", ".")

    # Step 4: Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

### Processing JD and Resume

In [353]:
def fix_numbers(text):
    # Fix CGPA like: 3 72 → 3.72
    text = re.sub(r'\b([0-4])\s+([0-9]{2})\b', r'\1.\2', text)

    # Fix format: 4 00 → 4.00
    text = re.sub(r'\b([0-9])\s+00\b', r'\1.00', text)

    return text

In [354]:
dataset_path = "/kaggle/working/dataset"

jd_text = ""
resume_text_list = []

for file in sorted(os.listdir(dataset_path)):
    file_path = os.path.join(dataset_path, file)

    if not os.path.isfile(file_path):
        continue

    # 1️⃣ Extract
    raw_text = extract_text(file_path)

    # 2️⃣ Clean
    cleaned_text = clean_text(raw_text)

    # 3️⃣ 🔥 FIX NUMBERS (AFTER CLEANING)
    cleaned_text = fix_numbers(cleaned_text)

    # Debug (VERY useful)
    # print("\nAFTER FIX:", cleaned_text[:200])

    # 4️⃣ Store
    if file.lower() == "jd.pdf":
        jd_text = cleaned_text
    else:
        resume_text_list.append(cleaned_text)

print("JD Length:", len(jd_text))
print("Number of Resumes:", len(resume_text_list))

JD Length: 1983
Number of Resumes: 7


Intermediate Output After Converting PDF into Text (Normal)

In [355]:
print("\n📄 JD TEXT PREVIEW:\n")
print(jd_text[:500])   # first 500 chars

print("\n📄 RESUME TEXT PREVIEW:\n")

for i, resume in enumerate(resume_text_list[:3]):  # first 3 resumes
    print(f"\n--- Resume {i+1} ---\n")
    print(resume[:300])   # first 300 chars


📄 JD TEXT PREVIEW:

role mts intern member technical staff job summary we are seeking a highly motivated and talented mts intern to join our team as an intern you will be responsible for web development automation and programming tasks you will work on a variety of projects collaborating with cross functional teams to deliver high quality software solutions this is an excellent opportunity to gain practical experience and enhance your skills in a fast paced environment join us and see where you can make an impact a

📄 RESUME TEXT PREVIEW:


--- Resume 1 ---

jainish parmar international institute of information technology bangalore cid 131 91 6356995938 jainishkumar parmar iiitb ac in cid 239 linkedin com in jainishparmar30 github com jainishparmar education international institute of information technology bangalore july 2024 present master of technolo

--- Resume 2 ---

aditya dave international institute of information technology bangalore cid 131 9316281792 aditya dave iiitb ac in

# Section 2: Text Normalization -> Standardize text for better comparison

In [356]:
nlp = spacy.load("en_core_web_sm")

In [357]:
abbreviation_map = {
    "ml": "machine learning",
    "ai": "artificial intelligence",
    "nlp": "natural language processing",
    "dl": "deep learning",
    "cv": "computer vision"
}

In [358]:
def normalize_text(text):
    text = text.lower()

    # expand abbreviations
    for abbr, full in abbreviation_map.items():
        text = re.sub(r'\b' + abbr + r'\b', full, text)

    doc = nlp(text)

    tokens = []
    i = 0

    while i < len(doc):
        token = doc[i]

        # 🔥 FIX: handle decimal numbers like 3 . 72
        if (
            i < len(doc) - 2 and
            doc[i].like_num and
            doc[i+1].text == "." and
            doc[i+2].like_num
        ):
            tokens.append(doc[i].text + "." + doc[i+2].text)
            i += 3
            continue

        if not token.is_stop and not token.is_punct:
            tokens.append(token.lemma_)

        i += 1

    return " ".join(tokens)

In [359]:
clean_jd_text = normalize_text(jd_text)
clean_resume_text = [normalize_text(r) for r in resume_text_list]

In [360]:
print("\nJD Preview:\n", clean_jd_text[:2000])

for i, r in enumerate(clean_resume_text[:2]):
    print(f"\nResume {i+1}:\n", r[:3000])


JD Preview:
 role mts intern member technical staff job summary seek highly motivated talented mts intern join team intern responsible web development automation programming task work variety project collaborate cross functional team deliver high quality software solution excellent opportunity gain practical experience enhance skill fast pace environment join impact netapp job requirement good programming skill c c python java golang understanding datum structure algorithm basic multi threading concept exposure nodejs javascript typescript web development frontend backend familiarity linux basic operating system fundamental experience python scripting automation simple testing framework like pyt basic understanding coursework exposure docker kubernete ci cd tool prefer require knowledge rest apis microservice basic distribute system concept plus familiarity cloud platform openstack cloudstack cloud technology nice interest datum science machine learning artificial intelligence tool pl

# Section 3: Information Extraction => Convert unstructured text → structured data

In [361]:
SKILL_SET = [
    "python", "java", "c", "c++", "sql", "javascript",
    "react", "node", "spring", "docker", "kubernetes",
    "aws", "machine learning", "deep learning",
    "nlp", "data science", "tensorflow", "pytorch",
    "linux", "git", "mongodb", "mysql"
]

def extract_skills(text):
    found_skills = []
    
    for skill in SKILL_SET:
        if skill in text:
            found_skills.append(skill)
    
    return list(set(found_skills))

In [362]:
def extract_education(text):
    education_keywords = [
        "btech", "mtech", "bachelor", "master",
        "phd", "computer science", "engineering"
    ]
    
    found = []
    
    for word in education_keywords:
        if word in text:
            found.append(word)
    
    return list(set(found))

In [363]:
def get_experience_section(text):
    text = text.lower()
    
    # start keywords
    start_keywords = ["experience", "work experience", "employment"]
    
    # end keywords
    end_keywords = ["education", "project", "skills", "certification"]
    
    start_idx = -1
    
    for key in start_keywords:
        if key in text:
            start_idx = text.find(key)
            break
    
    if start_idx == -1:
        return ""
    
    # find nearest end section
    end_idx = len(text)
    
    for key in end_keywords:
        idx = text.find(key, start_idx + 1)
        if idx != -1:
            end_idx = min(end_idx, idx)
    
    return text[start_idx:end_idx]


def extract_experience(text):
    exp_section = get_experience_section(text)
    
    years = re.findall(r'\b(20\d{2})\b', exp_section)
    years = [int(y) for y in years]
    
    current_year = datetime.now().year
    
    # keep realistic years
    years = [y for y in years if 2010 <= y <= current_year + 1]
    
    years = list(set(years))
    
    if len(years) >= 2:
        return max(years) - min(years)
    
    return 0

In [364]:
DOMAIN_MAP = {
    "machine learning": "AI/ML",
    "deep learning": "AI/ML",
    "nlp": "AI/ML",
    "data science": "AI/ML",
    "react": "Web Development",
    "node": "Web Development",
    "spring": "Backend",
    "docker": "DevOps",
    "kubernetes": "DevOps"
}

def identify_domain(text):
    domains = set()

    if any(k in text for k in ["machine learning", "deep learning", "nlp", "data science"]):
        domains.add("AI/ML")

    if any(k in text for k in ["react", "node", "frontend", "backend", "javascript"]):
        domains.add("Web Development")

    if any(k in text for k in ["docker", "kubernetes", "ci cd", "jenkins"]):
        domains.add("DevOps")

    if any(k in text for k in ["sql", "mysql", "mongodb", "data warehouse"]):
        domains.add("Data Engineering")

    if not domains:
        domains.add("General")

    return list(domains)

In [365]:
JD_struct = {
    "skills": extract_skills(jd_text),
    "education": extract_education(jd_text),
    "experience": extract_experience(jd_text),
    "domain": identify_domain(jd_text)
}

print("JD Struct:\n", JD_struct)

JD Struct:
 {'skills': ['c', 'machine learning', 'linux', 'python', 'java', 'javascript', 'data science', 'docker', 'node', 'kubernetes'], 'education': ['engineering', 'computer science', 'master'], 'experience': 0, 'domain': ['Web Development', 'AI/ML', 'DevOps']}


In [366]:
Resume_struct = []

for resume in resume_text_list:
    struct = {
        "skills": extract_skills(resume),
        "education": extract_education(resume),
        "experience": extract_experience(resume),  # FIXED
        "domain": identify_domain(resume)
    }
    
    Resume_struct.append(struct)

In [367]:
for i, struct in enumerate(Resume_struct):
    print(f"\n📄 Resume {i+1}")
    print("-" * 40)
    
    print(f"Skills     : {', '.join(struct['skills'])}")
    print(f"Education  : {', '.join(struct['education'])}")
    print(f"Experience : {struct['experience']} years")
    print(f"Domain     : {struct['domain']}")


📄 Resume 1
----------------------------------------
Skills     : react, c, mongodb, machine learning, aws, linux, git, python, mysql, java, javascript, spring, docker, node, kubernetes, sql
Education  : engineering, bachelor, computer science, master
Experience : 0 years
Domain     : ['Web Development', 'AI/ML', 'DevOps', 'Data Engineering']

📄 Resume 2
----------------------------------------
Skills     : react, c, git, linux, python, mysql, java, javascript, spring, docker, sql, nlp
Education  : engineering, computer science
Experience : 1 years
Domain     : ['Web Development', 'AI/ML', 'DevOps', 'Data Engineering']

📄 Resume 3
----------------------------------------
Skills     : react, c, mongodb, machine learning, git, python, mysql, java, javascript, spring, sql
Education  : bachelor, computer science, master
Experience : 0 years
Domain     : ['Web Development', 'AI/ML', 'Data Engineering']

📄 Resume 4
----------------------------------------
Skills     : c, git, linux, mysql, d

In [368]:
print("\n========== JD STRUCT ==========\n")

print("Skills:")
print(JD_struct["skills"])

print("\nEducation:")
print(JD_struct["education"])

print("\nExperience:")
print(JD_struct["experience"])

print("\nDomain:")
print(JD_struct["domain"])


========== JD STRUCT ==========

Skills:
['c', 'machine learning', 'linux', 'python', 'java', 'javascript', 'data science', 'docker', 'node', 'kubernetes']

Education:
['engineering', 'computer science', 'master']

Experience:
0

Domain:
['Web Development', 'AI/ML', 'DevOps']
